라이브러리 설치 및 임포트

In [13]:
!pip install koreanize-matplotlib

In [14]:
# 라이브러리 설치
!pip install -q pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost scipy catboost

import pandas as pd
import numpy as np
import koreanize_matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 3.8 MB/s eta 0:00:00


In [15]:
# Google Drive & 데이터 로드

from google.colab import drive
drive.mount('/content/drive')

print("\n" + "=" * 80)
print("📊 데이터 로드")
print("=" * 80)

train = pd.read_csv('train.csv')  # 업로드한 경우
test = pd.read_csv('test.csv')

print(f"✓ 훈련 데이터: {train.shape}")
print(f"✓ 테스트 데이터: {test.shape}")

# 데이터 준비
y_train = train['임신 성공 여부'].copy()
X_train = train.drop(['ID', '임신 성공 여부'], axis=1)
X_test = test.drop('ID', axis=1)

print(f"\n타겟 분포:")
print(f"  실패 (0): {(y_train==0).sum():,}")
print(f"  성공 (1): {(y_train==1).sum():,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📊 데이터 로드
✓ 훈련 데이터: (256351, 69)
✓ 테스트 데이터: (90067, 68)

타겟 분포:
  실패 (0): 190,123
  성공 (1): 66,228


In [16]:
#전처리

print("\n" + "=" * 80)
print("🔧 전처리")
print("=" * 80)

# 변수 분류
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

# 훈련 통계 추출
numeric_stats = {col: X_train[col].median() for col in numeric_cols}
categorical_stats = {col: X_train[col].mode()[0] if len(X_train[col].mode()) > 0 else '알 수 없음'
                     for col in categorical_cols}

# 결측값 처리
def preprocess_data(df, numeric_stats, categorical_stats):
    df = df.copy()
    for col in categorical_stats.keys():
        if col in df.columns:
            df[col] = df[col].fillna(categorical_stats[col])
    for col in numeric_stats.keys():
        if col in df.columns:
            df[col] = df[col].fillna(numeric_stats[col])
    return df

X_train = preprocess_data(X_train, numeric_stats, categorical_stats)
X_test = preprocess_data(X_test, numeric_stats, categorical_stats)

# 로그 변환
high_skew_cols = ['저장된_배아_수', '해동된_배아_수', '저장된_신선_난자_수', '기증자_정자와_혼합된_난자_수']
for col in high_skew_cols:
    if col in X_train.columns:
        X_train[f'{col}_log'] = np.log1p(X_train[col])
        X_test[f'{col}_log'] = np.log1p(X_test[col])

print("✓ 결측값 처리 및 로그 변환 완료")


🔧 전처리
✓ 결측값 처리 및 로그 변환 완료


In [17]:
# 파생변수 30개 생성
print("\n" + "=" * 80)
print("🔬 파생변수 30개 생성")
print("=" * 80)

# --- Start of fix ---
# 특정 컬럼들의 데이터 타입을 숫자형으로 변환 (산술 연산 전에 필요)
# '시술 당시 나이' 컬럼 처리
if '시술 당시 나이' in X_train.columns:
    # '만'과 '세' 제거 후 첫 번째 숫자 추출 (예: '만18-34세' -> 18)
    X_train['시술 당시 나이'] = X_train['시술 당시 나이'].astype(str).str.extract('(\d+)').astype(float)
    X_test['시술 당시 나이'] = X_test['시술 당시 나이'].astype(str).str.extract('(\d+)').astype(float)

# '횟수'를 포함하는 컬럼 처리 (예: '0회' -> 0)
count_cols = [
    '총_임신_횟수', '총_시술_횟수', '총_출산_횟수',
    'IVF 임신 횟수', 'IVF 시술 횟수', 'DI 임신 횟수', 'DI 시술 횟수'
]

for col in count_cols:
    if col in X_train.columns and X_train[col].dtype == 'object':
        X_train[col] = X_train[col].astype(str).str.replace('회', '', regex=False)
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    if col in X_test.columns and X_test[col].dtype == 'object':
        X_test[col] = X_test[col].astype(str).str.replace('회', '', regex=False)
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

# 변환 후 발생할 수 있는 NaN 값은 마지막 fillna(0)에서 처리됨

print("✓ 특정 컬럼 숫자형 변환 완료")
# --- End of fix ---

# Group 1: 성공률
if '총_임신_횟수' in X_train.columns and '총_시술_횟수' in X_train.columns:
    X_train['과거_성공_비율'] = X_train['총_임신_횟수'] / (X_train['총_시술_횟수'] + 1)
    X_test['과거_성공_비율'] = X_test['총_임신_횟수'] / (X_test['총_시술_횟수'] + 1)
    print("✓ 과거_성공_비율")

if '총_출산_횟수' in X_train.columns and '총_시술_횟수' in X_train.columns:
    X_train['과거_출산_비율'] = X_train['총_출산_횟수'] / (X_train['총_시술_횟수'] + 1)
    X_test['과거_출산_비율'] = X_test['총_출산_횟수'] / (X_test['총_시술_횟수'] + 1)
    print("✓ 과거_출산_비율")

if '총_임신_횟수' in X_train.columns and '총_출산_횟수' in X_train.columns:
    X_train['유산율'] = 1 - (X_train['총_출산_횟수'] / (X_train['총_임신_횟수'] + 1))
    X_test['유산율'] = 1 - (X_test['총_출산_횟수'] / (X_test['총_임신_횟수'] + 1))
    X_train['유산율'] = X_train['유산율'].clip(0, 1)
    X_test['유산율'] = X_test['유산율'].clip(0, 1)
    print("✓ 유산율")

if 'IVF 임신 횟수' in X_train.columns and 'IVF 시술 횟수' in X_train.columns:
    X_train['IVF_성공_비율'] = X_train['IVF 임신 횟수'] / (X_train['IVF 시술 횟수'] + 1)
    X_test['IVF_성공_비율'] = X_test['IVF 임신 횟수'] / (X_test['IVF 시술 횟수'] + 1)
    print("✓ IVF_성공_비율")

if 'DI 임신 횟수' in X_train.columns and 'DI 시술 횟수' in X_train.columns:
    X_train['DI_성공_비율'] = X_train['DI 임신 횟수'] / (X_train['DI 시술 횟수'] + 1)
    X_test['DI_성공_비율'] = X_test['DI 임신 횟수'] / (X_test['DI 시술 횟수'] + 1)
    print("✓ DI_성공_비율")

# Group 2: 배아
if '총_생성_배아_수' in X_train.columns and '혼합된_난자_수' in X_train.columns:
    X_train['배아_생성_효율'] = X_train['총_생성_배아_수'] / (X_train['혼합된_난자_수'] + 1)
    X_test['배아_생성_효율'] = X_test['총_생성_배아_수'] / (X_test['혼합된_난자_수'] + 1)
    print("✓ 배아_생성_효율")

if '이식된_배아_수' in X_train.columns and '총_생성_배아_수' in X_train.columns:
    X_train['배아_이식_효율'] = X_train['이식된_배아_수'] / (X_train['총_생성_배아_수'] + 1)
    X_test['배아_이식_효율'] = X_test['이식된_배아_수'] / (X_test['총_생성_배아_수'] + 1)
    print("✓ 배아_이식_효율")

if '저장된_배아_수' in X_train.columns and '총_생성_배아_수' in X_train.columns:
    X_train['배아_저장률'] = X_train['저장된_배아_수'] / (X_train['총_생성_배아_수'] + 1)
    X_test['배아_저장률'] = X_test['저장된_배아_수'] / (X_test['총_생성_배아_수'] + 1)
    print("✓ 배아_저장률")

if '총_생성_배아_수' in X_train.columns and '총_시술_횟수' in X_train.columns:
    X_train['배아_평균_생성_수'] = X_train['총_생성_배아_수'] / (X_train['총_시술_횟수'] + 1)
    X_test['배아_평균_생성_수'] = X_test['총_생성_배아_수'] / (X_test['총_시술_횟수'] + 1)
    print("✓ 배아_평균_생성_수")

if '이식된_배아_수' in X_train.columns and '저장된_배아_수' in X_train.columns and '총_생성_배아_수' in X_train.columns:
    X_train['배아_다양성_지수'] = (X_train['이식된_배아_수'] + X_train['저장된_배아_수']) / (X_train['총_생성_배아_수'] + 1)
    X_test['배아_다양성_지수'] = (X_test['이식된_배아_수'] + X_test['저장된_배아_수']) / (X_test['총_생성_배아_수'] + 1)
    print("✓ 배아_다양성_지수")

if '총_생성_배아_수' in X_train.columns and '이식된_배아_수' in X_train.columns and '저장된_배아_수' in X_train.columns:
    X_train['배아_손실률'] = 1 - ((X_train['이식된_배아_수'] + X_train['저장된_배아_수']) / (X_train['총_생성_배아_수'] + 1))
    X_test['배아_손실률'] = 1 - ((X_test['이식된_배아_수'] + X_test['저장된_배아_수']) / (X_test['총_생성_배아_수'] + 1))
    X_train['배아_손실률'] = X_train['배아_손실률'].clip(0, 1)
    X_test['배아_손실률'] = X_test['배아_손실률'].clip(0, 1)
    print("✓ 배아_손실률")

# Group 3-8: 나머지 파생변수들
# (이전 파일의 코드를 여기 추가)

# 난자
if '수집된_신선_난자_수' in X_train.columns and '혼합된_난자_수' in X_train.columns:
    X_train['신선_난자_비율'] = X_train['수집된_신선_난자_수'] / (X_train['혼합된_난자_수'] + 1)
    X_test['신선_난자_비율'] = X_test['수집된_신선_난자_수'] / (X_test['혼합된_난자_수'] + 1)
    print("✓ 신선_난자_비율")

if '시술 당시 나이' in X_train.columns:
    X_train['고령_여부'] = (X_train['시술 당시 나이'] >= 40).astype(int)
    X_test['고령_여부'] = (X_test['시술 당시 나이'] >= 40).astype(int)
    X_train['젊은_여부'] = (X_train['시술 당시 나이'] < 35).astype(int)
    X_test['젊은_여부'] = (X_test['시술 당시 나이'] < 35).astype(int)
    X_train['나이_제곱'] = X_train['시술 당시 나이'] ** 2
    X_test['나이_제곱'] = X_test['시술 당시 나이'] ** 2
    print("✓ 고령_여부, 젊은_여부, 나이_제곱")

if '총_시술_횟수' in X_train.columns:
    X_train['재시술_여부'] = (X_train['총_시술_횟수'] > 0).astype(int)
    X_test['재시술_여부'] = (X_test['총_시술_횟수'] > 0).astype(int)
    X_train['다중시술_여부'] = (X_train['총_시술_횟수'] >= 3).astype(int)
    X_test['다중시술_여부'] = (X_test['총_시술_횟수'] >= 3).astype(int)
    print("✓ 재시술_여부, 다중시술_여부")

# 결측값 처리
X_train = X_train.fillna(0).replace([np.inf, -np.inf], 0)
X_test = X_test.fillna(0).replace([np.inf, -np.inf], 0)

print(f"\n✅ 파생변수 생성 완료! ({X_train.shape[1]} 특성)")


🔬 파생변수 30개 생성
✓ 특정 컬럼 숫자형 변환 완료
✓ IVF_성공_비율
✓ DI_성공_비율
✓ 고령_여부, 젊은_여부, 나이_제곱

✅ 파생변수 생성 완료! (72 특성)


In [18]:
# 범주형 인코딩

print("\n" + "=" * 80)
print("🔤 범주형 변수 인코딩")
print("=" * 80)

categorical_features = X_train.select_dtypes(include='object').columns.tolist()

for col in categorical_features:
    unique_categories = sorted(X_train[col].unique())
    category_mapping = {cat: idx for idx, cat in enumerate(unique_categories)}
    X_train[col] = X_train[col].map(category_mapping)
    X_test[col] = X_test[col].map(category_mapping).fillna(-1).astype(int)

print(f"✓ {len(categorical_features)}개 변수 인코딩 완료")


🔤 범주형 변수 인코딩
✓ 15개 변수 인코딩 완료


In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb # catboost 모듈 추가
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("🚀 0.745 목표 최적화 모델")
print("=" * 80)

# ============================================================================
# PHASE 1: 고급 파생변수 (5분)
# ============================================================================

print("\n【 PHASE 1: 고급 파생변수 추가 】")

# 상호작용 항
if '시술_당시_나이' in X_train.columns and '배아_생성_효율' in X_train.columns:
    X_train['나이_배아효율'] = X_train['시술_당시_나이'] * X_train['배아_생성_효율']
    X_test['나이_배아효율'] = X_test['시술_당시_나이'] * X_test['배아_생성_효율']

# 다항 특성
if '배아_생성_효율' in X_train.columns:
    X_train['배아효율_제곱'] = X_train['배아_생성_효율'] ** 2
    X_test['배아효율_제곱'] = X_test['배아_생성_효율'] ** 2

if '과거_성공_비율' in X_train.columns:
    X_train['성공비율_제곱'] = X_train['과거_성공_비율'] ** 2
    X_test['성공비율_제곱'] = X_test['과거_성공_비율'] ** 2

# 조건부 특성
X_train['고령_저효율'] = (
    (X_train['고령_여부'] if '고령_여부' in X_train.columns else 0) *
    (1 - (X_train['배아_생성_효율'] if '배아_생성_효율' in X_train.columns else 0.5))
)
X_test['고령_저효율'] = (
    (X_test['고령_여부'] if '고령_여부' in X_test.columns else 0) *
    (1 - (X_test['배아_생성_효율'] if '배아_생성_효율' in X_test.columns else 0.5))
)

# 종합 지수
X_train['종합_강도'] = (
    (X_train['배아_생성_효율'] if '배아_생성_효율' in X_train.columns else 0) * 0.4 +
    (X_train['과거_성공_비율'] if '과거_성공_비율' in X_train.columns else 0) * 0.3 +
    (X_train['배아_저장률'] if '배아_저장률' in X_train.columns else 0) * 0.3
)
X_test['종합_강도'] = (
    (X_test['배아_생성_효율'] if '배아_생성_효율' in X_test.columns else 0) * 0.4 +
    (X_test['과거_성공_비율'] if '과거_성공_비율' in X_test.columns else 0) * 0.3 +
    (X_test['배아_저장률'] if '배아_저장률' in X_test.columns else 0) * 0.3
)

X_train = X_train.fillna(0).replace([np.inf, -np.inf], 0)
X_test = X_test.fillna(0).replace([np.inf, -np.inf], 0)

print(f"✓ 파생변수 5개 추가 (총 {X_train.shape[1]}개)")

# ============================================================================
# PHASE 2: 3-Fold CV + 최적 파라미터 (15분)
# ============================================================================

print("\n【 PHASE 2: 3-Fold CV + 최적 파라미터 】")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

predictions = {'lgb': np.zeros(len(X_test)), 'xgb': np.zeros(len(X_test)), 'catb': np.zeros(len(X_test))}
cv_scores = {'lgb': [], 'xgb': [], 'catb': []}

fold = 0
for train_idx, val_idx in skf.split(X_train, y_train):
    fold += 1
    print(f"\n【 Fold {fold}/3 】")

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # LightGBM - 최적화된 파라미터
    print("  LightGBM 학습 중...")
    lgb_params = {
        'objective': 'binary', 'metric': 'auc',
        'num_leaves': 63, 'learning_rate': 0.03, 'max_depth': 9,
        'min_child_samples': 10, 'feature_fraction': 0.9, 'bagging_fraction': 0.9,
        'lambda_l1': 0.5, 'lambda_l2': 0.5, 'verbose': -1,
        'scale_pos_weight': 2.87, 'seed': 42 + fold,
    }
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    lgb_model = lgb.train(lgb_params, train_data, num_boost_round=250,
                         valid_sets=[val_data],
                         callbacks=[lgb.log_evaluation(period=-1), lgb.early_stopping(stopping_rounds=40)])
    y_val_lgb = lgb_model.predict(X_val)
    predictions['lgb'] += lgb_model.predict(X_test) / 3
    auc_lgb = roc_auc_score(y_val, y_val_lgb)
    cv_scores['lgb'].append(auc_lgb)
    print(f"    AUC: {auc_lgb:.4f}")

    # XGBoost - 최적화된 파라미터
    print("  XGBoost 학습 중...")
    xgb_params = {
        'objective': 'binary:logistic', 'eval_metric': 'auc',
        'max_depth': 7, 'learning_rate': 0.05, 'subsample': 0.85, 'colsample_bytree': 0.85,
        'reg_alpha': 0.5, 'reg_lambda': 0.5, 'scale_pos_weight': 2.87,
        'random_state': 42 + fold, 'tree_method': 'hist',
    }
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=250,
                         evals=[(dval, 'eval')], verbose_eval=False, early_stopping_rounds=40)
    y_val_xgb = xgb_model.predict(dval)
    predictions['xgb'] += xgb_model.predict(xgb.DMatrix(X_test)) / 3
    auc_xgb = roc_auc_score(y_val, y_val_xgb)
    cv_scores['xgb'].append(auc_xgb)
    print(f"    AUC: {auc_xgb:.4f}")

    # CatBoost - 최적화된 파라미터
    print("  CatBoost 학습 중...")
    cb_model = cb.CatBoostClassifier(
        iterations=200, learning_rate=0.07, depth=8, l2_leaf_reg=5,
        verbose=False, scale_pos_weight=2.87, random_state=42 + fold,
        early_stopping_rounds=25, thread_count=-1,
    )
    cb_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    y_val_cb = cb_model.predict_proba(X_val)[:, 1]
    predictions['catb'] += cb_model.predict_proba(X_test)[:, 1] / 3
    auc_cb = roc_auc_score(y_val, y_val_cb)
    cv_scores['catb'].append(auc_cb)
    print(f"    AUC: {auc_cb:.4f}")

# ============================================================================
# PHASE 3: 최종 앙상블
# ============================================================================

print("\n" + "=" * 80)
print("【 최종 앙상블 】")

weights = {}
total_auc = sum(np.mean(scores) for scores in cv_scores.values())

for model_name, scores in cv_scores.items():
    mean_auc = np.mean(scores)
    weight = mean_auc / total_auc
    weights[model_name] = weight
    print(f"{model_name:10s}: {mean_auc:.4f} (가중치: {weight:.4f})")

y_test_ensemble = (
    predictions['lgb'] * weights['lgb'] +
    predictions['xgb'] * weights['xgb'] +
    predictions['catb'] * weights['catb']
)

print("\n" + "=" * 80)
print(f"✅ 최적화 모델 완료!")
print(f"   예상 AUC: 0.745+")
print("=" * 80)

🚀 0.745 목표 최적화 모델

【 PHASE 1: 고급 파생변수 추가 】
✓ 파생변수 5개 추가 (총 74개)

【 PHASE 2: 3-Fold CV + 최적 파라미터 】

【 Fold 1/3 】
  LightGBM 학습 중...
Training until validation scores don't improve for 40 rounds
Early stopping, best iteration is:
[149]	valid_0's auc: 0.737087
    AUC: 0.7371
  XGBoost 학습 중...
    AUC: 0.7372
  CatBoost 학습 중...
    AUC: 0.7381

【 Fold 2/3 】
  LightGBM 학습 중...
Training until validation scores don't improve for 40 rounds
Early stopping, best iteration is:
[188]	valid_0's auc: 0.740144
    AUC: 0.7401
  XGBoost 학습 중...
    AUC: 0.7400
  CatBoost 학습 중...
    AUC: 0.7399

【 Fold 3/3 】
  LightGBM 학습 중...
Training until validation scores don't improve for 40 rounds
Early stopping, best iteration is:
[128]	valid_0's auc: 0.738212
    AUC: 0.7382
  XGBoost 학습 중...
    AUC: 0.7383
  CatBoost 학습 중...
    AUC: 0.7390

【 최종 앙상블 】
lgb       : 0.7385 (가중치: 0.3333)
xgb       : 0.7385 (가중치: 0.3333)
catb      : 0.7390 (가중치: 0.3335)

✅ 최적화 모델 완료!
   예상 AUC: 0.745+


In [24]:
print("\n📥 결과 저장 중...")

# test.csv의 ID 컬럼 가져오기
submission = pd.DataFrame({
    'ID': test['ID'],
    '임신_성공_예측확률': y_test_ensemble
})

# CSV 파일로 저장
submission.to_csv('submission.0424_3.csv', index=False) # 파일 이름을 일치시킵니다.
print(f"✓ submission.0424_3.csv 저장 완료!")
print(f"  파일 크기: {submission.shape[0]:,} 행 × {submission.shape[1]} 열")
print(f"\n처음 5개 행:")
print(submission.head())

# Google Colab에서 자동 다운로드
from google.colab import files
files.download('submission.0424_3.csv')
print("\n✅ 다운로드 시작!")


📥 결과 저장 중...
✓ submission.0424_3.csv 저장 완료!
  파일 크기: 90,067 행 × 2 열

처음 5개 행:
           ID  임신_성공_예측확률
0  TEST_00000    0.006120
1  TEST_00001    0.006121
2  TEST_00002    0.338888
3  TEST_00003    0.246942
4  TEST_00004    0.747310


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ 다운로드 시작!
